In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from google import genai
genai_client = genai.Client()

In [ ]:
def llm(prompt):
    response = genai_client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )

    return response.text

In [ ]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

edit on GitHub
#Cloud alternatives with GPU
Check the quota and reset cycle carefully. Is the free hours limit per month or per week? Usually, if you change the configuration, the free hours quota might also be adjusted, or it might be billed separately.

Potential options include:

Google Colab
Kaggle
Databricks (possibly)
Consider using GPTs to discover more options. Be aware that some platforms might have restrictions on what you can and cannot install, so ensure to read what is included in the free vs paid tier.
'''

In [ ]:
question = 'I just discovered the course. Can I join now?'
prompt = f'''
Your task is to answer questions from the course participants"
based pn the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''
answer = llm(prompt)
print(answer)

In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [ ]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [ ]:
courses_raw

In [ ]:
documents = []

url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

In [ ]:
documents[500]

In [ ]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [ ]:
search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)
search_results

In [ ]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [ ]:
search_results = search(question)
search_results

In [ ]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants"
based pn the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [ ]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [ ]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: '+ doc['question'] )
        lines.append('A: '+ doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question, context=context
    )
    return prompt.strip()

In [ ]:
prompt = build_prompt(question, search_results)
print(prompt)

In [ ]:
response = genai_client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)
response.text

In [ ]:
print(response.model_dump_json(indent=2))

In [ ]:
token_response = genai_client.models.count_tokens(
    model="gemini-3.5-flash",
    contents=prompt
)

print(f"Total input tokens: {token_response.total_tokens}")

In [ ]:
metadata = response.usage_metadata
print(f"Prompt Tokens: {metadata.prompt_token_count}")
print(f"Output Tokens: {metadata.candidates_token_count}")
print(f"Total Tokens: {metadata.total_token_count}")

In [ ]:
input_cost = (token_response.total_tokens / 1_000_000) * 2.70
output_cost = (metadata.candidates_token_count / 1_000_000) * 16.20
total_cost_usd = input_cost + output_cost

print(f"Total Request Cost: ${total_cost_usd:.6f} USD")

In [ ]:
message_history = [
    {
        'role': 'system',
        'parts': [
            {"text": INSTRUCTIONS}
        ],
    },
    {
        'role': 'user',
        'parts': [
            {"text": prompt}
        ],
    }
]

response = genai_client.models.generate_content(
    model="gemini-3.5-flash",
    contents=message_history
)

In [ ]:
response.text

In [ ]:
def llm(instructions, user_prompt, model="gemini-3.5-flash"):
    message_history = [
        {
            'role': 'system',
            'parts': [
                {"text": instructions}
            ],
        },
        {
            'role': 'user',
            'parts': [
                {"text": user_prompt}
            ],
        }
    ]

    response = genai_client.models.generate_content(
        model=model,
        contents=message_history
    )

    return response.text

In [ ]:
def rag(query, model="gemini-3.5-flash"):
    search_results = search(query)
    user_prompt = build_prompt(query, search_results)
    return llm(INSTRUCTIONS, user_prompt, model)

In [ ]:
answer = rag(question)
print(answer)